# EDSS 취업데이터 2022–2023 잠정 교차표

## tl;dr

2023 대학 415개와 일반대학원 185개를 2022 개방ID에 비교했다. 학과 수 차이 0~2, 학과명 정확/고중첩, 동일 지역 내 양방향 유일 최상위 조건을 모두 만족한 66개만 잠정 교차표에 남겼다. 전문·특수대학원은 2022와 집계 단위가 달라 별도 모대학 그룹 진단으로 분리했다.

## Context & Methods

- 비교 집단: 2023 `대학`과 `일반대학원`
- 후보 조건: 같은 지역·구분에서 고유 학과 수 차이 0~2
- 강한 증거: 학과 집합 정확 일치(3개 이상) 또는 Jaccard 0.80 이상·작은 집합 포괄률 0.90 이상
- 수락 조건: 2023 학교→2022 개방ID와 2022 개방ID→2023 학교가 모두 유일한 최상위
- 별도 교차검증: 기존 2023 대학·대학원 개방ID 후보 자료
- 전문·특수대학원: 대학명 최장 접두어와 독립 대학원대학교 규칙으로만 모대학 그룹 진단

### Assumptions

2022 대학원 개방ID의 비교 가능한 2023 집단은 지역별 학교 수가 거의 같은 일반대학원이라고 본다. 잠정 수락 행도 공식 교차표가 아니며 원본이나 정식 패널에 쓰지 않는다.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
from collections import Counter
import pandas as pd

ROOT = Path('/Users/joocheol/Documents/ChatGPT/EDSS')
RAW_ROOT = Path('/Users/joocheol/Documents/GitHub/edss/data/raw/edss')
SCRIPT = ROOT / 'scripts/build_edss_employment_2022_2023_provisional_crosswalk.py'
SUMMARY = ROOT / 'data/metadata/edss_employment_2022_2023_provisional_crosswalk.json'
REVIEW = ROOT / 'data/processed/edss/restricted/derived/employment_2022_2023_reciprocal_review.csv'
CROSSWALK = ROOT / 'data/processed/edss/restricted/derived/employment_2022_2023_provisional_crosswalk.csv'
REGION = ROOT / 'data/metadata/edss_employment_2022_2023_comparable_region_counts.csv'
PARENTS = ROOT / 'data/metadata/edss_employment_2023_non_general_graduate_parent_groups.csv'

## Data

원본 ZIP과 기존 교차검증 후보 자료의 체크섬을 기록하고 전체 분석을 다시 실행한다.

In [2]:
run = subprocess.run(
    ['python3', str(SCRIPT), '--raw-root', str(RAW_ROOT)],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
region = pd.read_csv(REGION, encoding='utf-8-sig')
summary['inputs']

{'source_2022': {'path': '/Users/joocheol/Documents/GitHub/edss/data/raw/edss/취업통계/0001_학생인적취업정보_13299/0001_학생인적취업정보_2022.zip',
  'sha256': 'e763b6a6848039be6c1e34975628600f772e36219a762610f8a6a59a1b694d13'},
 'source_2023': {'path': '/Users/joocheol/Documents/GitHub/edss/data/raw/edss/취업통계/0001_학생인적취업정보_13300/0001_학생인적취업정보_2023.zip',
  'sha256': '1ace9e82663a816e00a66a35d2b76980e79c708d54abd390564d3d90dcdbe4eb'},
 'bridge': {'path': 'data/metadata/edss_school_year_bridge.csv',
  'sha256': 'edfbdbf2cc9c20da00f47d059d668eb8fa4469fae79c6c3105bb356ab644a27d'},
 'enrollment_candidates': {'path': 'data/metadata/edss_employment_enrollment_open_id_candidates.csv',
  'sha256': '3666fa22e265a1363050ca72dc9fe48458c6d648708a4d0660c52263a6e0bcaa'},
 'graduate_candidates': {'path': 'data/metadata/edss_graduate_open_id_candidates.csv',
  'sha256': '9a933f94eb7e9cfb92a5207ac32c49907bcf045c993305fec3768d88ed6ab932'}}

## Results

학교명과 개방ID는 출력하지 않고 집계만 표시한다.

In [3]:
status = pd.DataFrame(summary['counts']['scope_status_counts']).fillna(0).astype(int).T
status

,accepted_reciprocal_exact,accepted_reciprocal_high_overlap,ambiguous_forward_top_score,no_count_tolerance_candidate,review_small_exact_signature,review_weak_best
대학,30,24,14,55,14,278
일반대학원,4,8,7,46,10,110


In [4]:
with CROSSWALK.open(encoding='utf-8-sig', newline='') as handle:
    crosswalk_rows = list(csv.DictReader(handle))
pd.DataFrame([
    {
        '구분': scope,
        '잠정 수락': sum(row['scope'] == scope for row in crosswalk_rows),
        '2023 개방ID 후보까지 연결': sum(row['scope'] == scope and bool(row['open_id_2023_candidate']) for row in crosswalk_rows),
    }
    for scope in ('대학', '일반대학원')
])

,구분,잠정 수락,2023 개방ID 후보까지 연결
0,대학,54,42
1,일반대학원,12,11


In [5]:
region_summary = []
for scope, frame in region.groupby('scope'):
    differences = frame['difference_2023_minus_2022'].abs()
    region_summary.append({
        '구분': scope,
        '2022 학교 출현 수': int(frame['schools_2022'].sum()),
        '2023 학교 수': int(frame['schools_2023'].sum()),
        '차이 2 이하 지역 수': int((differences <= 2).sum()),
        '지역별 최대 절대차': int(differences.max()),
    })
pd.DataFrame(region_summary)

,구분,2022 학교 출현 수,2023 학교 수,차이 2 이하 지역 수,지역별 최대 절대차
0,대학,386,415,14,16
1,일반대학원,183,185,17,1


In [6]:
pd.DataFrame([{
    '전문·특수대학원 학교명 단위': summary['counts']['non_general_graduate_identity_count'],
    '모대학 그룹 행': summary['counts']['parent_group_row_count'],
    **summary['counts']['parent_resolution_method_counts'],
}])

,전문·특수대학원 학교명 단위,모대학 그룹 행,longest_undergraduate_name_prefix,standalone_graduate_university,unresolved_parent
0,1019,262,975,43,1


### Validation

행 수, 상호 유일성, 체크섬과 정식 매핑 미작성 조건을 독립적으로 확인한다.

In [7]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with REVIEW.open(encoding='utf-8-sig', newline='') as handle:
    review_rows = list(csv.DictReader(handle))
assert len(review_rows) == summary['counts']['review_identity_count'] == 600
assert len(crosswalk_rows) == summary['counts']['accepted_provisional_count']
assert len({(r['scope'], r['province'], r['school_name_2023']) for r in crosswalk_rows}) == len(crosswalk_rows)
assert len({(r['scope'], r['province'], r['open_id_2022']) for r in crosswalk_rows}) == len(crosswalk_rows)
with_2023 = [r for r in crosswalk_rows if r['open_id_2023_candidate']]
assert len({(r['scope'], r['province'], r['open_id_2023_candidate']) for r in with_2023}) == len(with_2023)
assert all(r['match_status'].startswith('accepted_') for r in crosswalk_rows)
assert all(r['canonical_mapping_written'] == 'false' for r in crosswalk_rows)
for key, path in [('review', REVIEW), ('crosswalk', CROSSWALK), ('parent_groups', PARENTS), ('region_counts', REGION)]:
    assert sha256(path) == summary['outputs'][key]['sha256']
{'review_rows': len(review_rows), 'accepted_rows': len(crosswalk_rows), 'with_2023_open_id': len(with_2023), 'validation': 'PASS'}

{'review_rows': 600,
 'accepted_rows': 66,
 'with_2023_open_id': 53,
 'validation': 'PASS'}

## Takeaways

- 일반대학원은 17개 지역 모두 2022와 2023 학교 수 차이가 0~1이어서 직접 비교 가능한 집단이다.
- 양방향 유일 최상위와 최소 3개 학과 조건을 적용해 대학 54개, 일반대학원 12개를 잠정 수락했다.
- 이 중 53개는 별도 EDSS 자료의 2023 개방ID 후보까지 연결되며, 나머지 13개는 학교명 연결만 강하다.
- 전문·특수대학원 1,019개 중 1,018개는 모대학 또는 독립 대학원대학교로 묶였지만 262개 그룹이라 2022의 183개 단위와 여전히 다르다.
- 따라서 전문·특수대학원은 정식 1:1 교차표에서 제외하고 별도 다대일 계층으로 유지한다.